In [1]:
import sys
sys.path.insert(0, '../lib')

In [2]:
import collections
import functools
import joblib
import os
import pathlib

import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.stats.multitest

import common_data

/projects/b1196/envs/serniczek/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
pd.options.display.max_columns = 200
pd.options.display.max_rows = 200
%config InlineBackend.figure_format = "retina"

In [ ]:
ROOT = common_data.DATA
BASE = ROOT / '05_pseudobulk/40a_serial'

In [ ]:
adata = sc.read_h5ad(common_data.SC_NORM)

In [6]:
sc_labels = pd.read_csv(common_data.SC_LABELS, index_col=0)

In [7]:
%%time
PSEUDOBULK_CELLS = 50
PSEUDOBULK_EXPR_IN_GROUP = 0.8
genes_to_keep = {}
for ct in adata.obs.Level_6.unique():
    genes = None
    samples = adata.obs.bal_barcode.unique()
    pseudobulks = []
    for sample in samples:
        idx = adata.obs.Level_6.eq(ct) & adata.obs.bal_barcode.eq(sample)
        if idx.sum() < PSEUDOBULK_CELLS:
            continue
        pseudobulks.append(adata.raw.X[idx, :].sum(axis=0).A1)
    if len(pseudobulks) == 0:
        continue
    pseudobulks = pd.DataFrame(pseudobulks, columns=adata.raw.var_names)
    expr_frac = (pseudobulks > 0).sum(axis=0) / pseudobulks.shape[0]
    group_genes = pseudobulks.columns[expr_frac.ge(PSEUDOBULK_EXPR_IN_GROUP)]
    if genes is None:
        genes = group_genes.to_numpy()
    else:
        genes = np.union1d(genes, group_genes)
    genes_to_keep[ct] = genes

CPU times: user 40.9 s, sys: 2.36 s, total: 43.2 s
Wall time: 43.3 s


In [8]:
{k: len(v) for k, v in genes_to_keep.items() if v is not None}

{'CD4 T cells': 9750,
 'CD8 T cells': 9953,
 'Mast cells': 7542,
 'NUPR1+ Macs': 11909,
 'B cells': 8308,
 'MRC1+C1QA+': 11319,
 'DC2': 10164,
 'MRC1+C1QA-': 11527,
 'Proliferating CD4 T cells': 10240,
 'Proliferating CD8 T cells': 9867,
 'Classical monocytes-2 IL1B': 9600,
 'Secretory cells': 11734,
 'Proliferating NUPR1+ Macs': 11181,
 'Tregs': 8430,
 'DC1': 9102,
 'Ionocytes': 13684,
 'Ciliated cells': 11508,
 'gdT cells': 8547,
 'Migratory DC': 7925,
 'Interstitial macrophages': 9062,
 'Hematopoietic stem cells': 11644,
 'Classical monocytes-1 CCR2': 8467,
 'pDC': 9233,
 'AT1 and AT2': 11920,
 'Proliferating plasma cells': 10351,
 'Non-classical monocytes': 9011,
 'Plasma cells': 9384,
 'Proliferating gdT cells': 9304}

In [9]:
genes_to_keep['Perivascular macrophages'] = genes_to_keep['Interstitial macrophages']

In [ ]:
PADJ_CUTOFF = 0.05
class ComparisonInfo:
    def __init__(self, control, condition, genes, genes_to_keep):
        self.control = control
        self.condition = condition
        self.genes_raw = genes
        self.filter_genes(genes_to_keep)

    def filter_genes(self, genes_to_keep):
        filtered_degs = self.genes_raw.loc[self.genes_raw.index.isin(genes_to_keep), :].copy()
        # use padj, see https://bioconductor.org/packages/release/bioc/vignettes/DESeq2/inst/doc/DESeq2.html#indfilttheory
        filtered_degs = filtered_degs.loc[filtered_degs.padj.notna()].copy()
        # recompute FDR correction on the filtered genes:
        filtered_degs['padj'] = statsmodels.stats.multitest.fdrcorrection(
            filtered_degs.pvalue,
            alpha=PADJ_CUTOFF
        )[1]
        # recompute gene status based on new `padj`
        filtered_degs['sign'] = ''
        filtered_degs.loc[
            filtered_degs.padj.lt(PADJ_CUTOFF)
            & filtered_degs.log2FoldChange.gt(0),
            'sign'
        ] = f'Up in {self.condition}'
        filtered_degs.loc[
            filtered_degs.padj.lt(PADJ_CUTOFF)
            & filtered_degs.log2FoldChange.lt(0),
            'sign'
        ] = f'Up in {self.control}'
        self.genes = filtered_degs


class CellTypeInfo:
    def __init__(self, path, task_info: common_data.TaskInfo, genes_to_keep):
        self.path = path
        self.task_info = task_info
        self.comparisons = []
        self.meta = pd.read_csv(path / 'meta.csv', index_col=0)
        self.name = self.meta.cell_type.values[0]

        self.load_comparisons(genes_to_keep)

    def load_comparisons(self, genes_to_keep):
        fname = 'degs.csv'
        if self.task_info.pathname == 'vap':
            fname = 'degs-vap.csv'
        if self.task_info.pathname == 'no-vap':
            fname = 'degs-no-vap.csv'
        if self.task_info.pathname == 'baseline':
            fname = 'degs-baseline.csv'
        for run in self.path.glob(f'**/{fname}'):
            self.comparisons.append(
                ComparisonInfo(
                    self.task_info.column_values[0],
                    self.task_info.column_values[1],
                    pd.read_csv(run, index_col=0),
                    genes_to_keep[self.name]
                )
            )

    @property
    def n_comparisons(self):
        return len(self.comparisons)

In [11]:
class TaskData:
    def __init__(self, task, task_info):
        self.task = task
        self.task_info = task_info
        self.info = {}

In [ ]:
%%time
data = {}
for task in ('timepoint2', 'vap', 'no-vap', 'baseline'):
    task_name = task
    column_values = ['first', 'second']
    if task == 'timepoint2':
        column_values = ['no_vap', 'vap']
    if task == 'baseline':
        column_values = ['no_vap', 'vap']
    task_info = common_data.TaskInfo(
        pathname=task,
        column='pathogens_coarse',
        column_values=column_values,
        split_column='blah'
    )
    task_data = TaskData(task, task_info)
    for cell_type_path in sorted(BASE.iterdir()):
        if cell_type_path.name.startswith('.') or cell_type_path.name.startswith('_'):
            continue
        if not cell_type_path.is_dir():
            continue
        if not (cell_type_path / 'meta.csv').exists():
            continue
        info = CellTypeInfo(cell_type_path, task_info, genes_to_keep)
        if info.n_comparisons > 0:
            task_data.info[cell_type_path.name] = info
    if len(task_data.info) > 0:
        data[task_name] = task_data

CPU times: user 1.72 s, sys: 110 ms, total: 1.83 s
Wall time: 4.35 s


In [ ]:
data['timepoint2'].info['NUPR1+_Macs'].comparisons[0].genes.sign.value_counts()

    11540
Name: sign, dtype: int64

In [ ]:
data['vap'].info['MRC1+C1QA+'].comparisons[0].genes.sign.value_counts()

                11032
Up in first         3
Up in second        1
Name: sign, dtype: int64

In [ ]:
joblib.dump(data, '40c_deg_data.joblib')

Save filtered DEGs as csv to run GSEA on them

In [ ]:
BASE = ROOT / '05_pseudobulk/40c_degs'
for _, task in data.items():
    for k, ct_info in task.info.items():
        comp = ct_info.comparisons[0]
        deg_path = BASE / task.task_info.pathname / k / 'degs.csv'
        if deg_path.exists():
            print(f'File {deg_path} already exists, skipping')
            continue
        # Threshold GSEA analysis to at least 1000 genes in comparison
        if comp.genes.shape[0] < 1000:
            continue
        ct_path = BASE / task.task_info.pathname / k
        os.makedirs(ct_path, exist_ok=True)
        comp.genes.sort_values('log2FoldChange').to_csv(deg_path)